In [ ]:
import numpy as np
import pandas as pd
import math
import os
import argparse



In [ ]:

sc = 'column30'  # maximum possible solar collector potential 
heat = 'column3'  # heating demand

# TES info
store = {'min': 500, # min TES volume in liters
         'residential': 1500, # max TES volume  for residential buildings in liters
         'non_residential': 3000, # max TES volume  for non-residential buildings in liters
         'cap': 70, # TES volume to capacity in kWh/m^3
         'q_l': 0.99 } # TES heat store efficiency, hour to hour 


min_sc_area = 10 # min installed solar collector area
sc_module_area  = 2.5 # solar collector module area
min_sc_num = math.floor(min_sc_area / sc_module_area) # min number of solar collector module for installation


In [ ]:
def cost(area):
    dic = {'cost': 0,
           'ins_cost': 0,
           'capex': 0,
           'opex': 0}
    
    if area != 0:
        dic['cost'] = 400 * area + 2500
        dic['ins_cost'] = 0.25 * dic['cost']
        dic['capex'] = dic['cost'] + dic['ins_cost']
        dic['opex'] = 3.5 * area + 147.5
    return dic
    
def vol(area, usage):
    v = 40 * area + 220
    
    return max(store['min'], min(v, store[usage]))

def store_cap(vol):
    return vol * store['cap'] / 1000

def gas(h): 
    h = h / 0.8
    if h <= 2000:
        return 6.00 * 12 + (20.47 + 2.226) * h / 100 
    elif h <= 10000:
        return 8.00 * 12 + (18.62 + 2.226) * h / 100
    elif h <= 30000:
        return 12.00 * 12 + (17.73 + 2.226)* h / 100
    elif h <= 100000:
        return 14.00 * 12 + (17.61 + 2.226) * h / 100
    elif h <= 300000:
        return 24.00 * 12 + (17.37 + 2.226) * h / 100
    else:
        return 60.00 * 12 + (17.2 + 2.226) * h / 100
    
def initial_constants(dic, area):    
    dic['gas_savings'] = dic['gas_cost']  - dic['remain_gas_cost']
    result = cost(area)
    dic['capex'] = result['capex']
    dic['opex'] = result['opex']

    return dic


def npv_cal(dic, i):
    gas_increase = 0.04492035398 # average gas increase rate from euro stat 
    interest_rate = 0.0337 # interest rate from bundesbank
    c = dic['gas_savings'] * (1 + gas_increase) ** i - dic['opex']
    cur = c / (1 + interest_rate)**i 
    return cur

def npv(dic, years):

    dic['n_0'] = -1 * dic['capex']
    dic['npv'] = dic['n_0']
    for i in range(1, years+1):
        dic['n_'+str(i)] = npv_cal(dic, i)
        dic['npv'] = dic['npv'] + dic['n_'+str(i)]

    return dic

def cal_battery(o_df, total_cap, scale):
    
    q_l = store['q_l']

    df = o_df.copy()  # Make a copy of the dataframe
    
    scaled_sc = 'scaled_sc'  # solar collector potential if installing num of solar collector + storage system
    o = 'over_supply'  # solar collector potential over supply
    us = 'under_supply'  # solar collector potential used for heating demand at the same time stamp
    need = 'need'  # remaining heating demand after using solar potential at the same time stamp
    battery = 'battery'  # heat stored in battery
    after_battery = 'heat_aft_battery'  # remaining demand after using solar potential and battery
    
    df[scaled_sc] = df[sc] * scale
    df[o] = np.where(df[scaled_sc] <= df[heat], 0,  df[scaled_sc] - df[heat])
    df[us] = np.where(df[o] == 0, df[scaled_sc], df[heat])
    df[battery] = 0
    df[need] = df[heat] - df[us]
    df[after_battery] = df[need]
    
    df.at[0, battery] = df.at[0, scaled_sc] - df.at[0, need]
    
    if df.at[0, battery] < 0: df.at[0, battery] = 0

    
    for i in range(1, len(df)):
        cur = df.at[i-1, battery] * q_l
        
        if df.at[i, o] == 0 and df.at[i, heat] > 0 and cur > 0:
            need_val = df.at[i, need]
            withdraw = min(need_val, cur)
            cur -= withdraw
            df.at[i, after_battery] = need_val - withdraw
        
        cur += df.at[i, o]
        cur = min(cur, total_cap)
        df.at[i, battery] = cur
        
    return df, df[after_battery].sum(), df[heat].sum() - df[after_battery].sum()

def install_systems(df, max_area, usage):    
    max_num = int(max_area // sc_module_area)
    original_num = max_area / sc_module_area
    
    best_dic = {'heat': df[heat].sum()}
    best_dic['gas_cost'] = gas(best_dic['heat'])
    best_dic['need_aft_battery'] = best_dic['heat']
    best_dic['remain_gas_cost'] = best_dic['gas_cost']
    best_dic['num_sc'] = 0
    best_dic['total_installed_area'] = 0
    best_dic['total_storage'] = 0
    best_dic['total_capacity'] = 0
    best_dic['saved_aft_battery'] = 0
    best_dic = initial_constants(best_dic, 0)
    best_dic = npv(best_dic, 25)


    
    return_df = df.copy()
    eff = df[sc].sum() / max_area

    min_area = 6812.5 / (5.675 * eff - 587.5)
    
    if max_num < min_sc_num or eff < 587.5 / 5.675 or min_area > max_area:
        return best_dic, df
    
    min_num = max(min_sc_num, math.floor(min_area / sc_module_area))
    
    for i in range(max_num, min_num-1, -1):
        total_area = i * sc_module_area
        battery_vol = vol(total_area, usage)
        total_cap = store_cap(battery_vol)
        
        temp_df, after_battery, saved = cal_battery(df, i, i/original_num)
        
        temp_dic = best_dic.copy()
        temp_dic['need_aft_battery'] = after_battery
        temp_dic['remain_gas_cost'] =  gas(temp_dic['need_aft_battery'])
        temp_dic['num_sc'] = i
        temp_dic['total_installed_area'] = total_area
        temp_dic['total_storage'] = battery_vol
        temp_dic['total_capacity'] = total_cap
        temp_dic = initial_constants(temp_dic, total_area)
        temp_dic['saved_aft_battery'] = saved
        
        temp_dic = npv(temp_dic, 25)

        if temp_dic['npv'] >= best_dic['npv']:
            best_dic = temp_dic.copy()
            return_df = temp_df.copy()
                
    return best_dic, return_df

def main(inp, outp):
    overview_df = pd.read_csv(inp)
    
    prefix = "/storage/"
    
    summary = {'iri': [],
               'num_sc': [],
               'total_installed_area': [],
               'total_capacity': [],
               'need_aft_battery': [],
               'saved_aft_battery': [],
               'capex': [],
               'opex': [],
               'npv': []}
    
    for j in range(0, 26):
        summary['n_'+str(j)] = []
    
    c = 0
    overview_df = overview_df[(overview_df['ps'] == 'y') | (overview_df['postcode'].notna())]
    overview_df = overview_df[overview_df['roof_area']>=min_sc_area]
    
    for i, row in overview_df.iterrows(): 
        
        if os.path.exists(prefix + row['tableName'] + ".csv"):
            summary['iri'].append(row['iri'])
    
            df = pd.read_csv(prefix + row['tableName'] + ".csv")
            
            if row['roof_area'] >= min_sc_area:
                usage = 'non_residential'
        
                if 'Residential' in row['usage'] or 'Domestic' in row['usage']:
                    usage = 'residential'
                    
                result, df_result = install_systems(df, row['roof_area'], usage)
                
                summary['num_sc'].append(result['num_sc'])
                summary['total_installed_area'].append(result['total_installed_area'])
                summary['total_capacity'].append(result['total_capacity'])
                summary['capex'].append(result['capex'])
                summary['opex'].append(result['opex'])
                summary['need_aft_battery'].append(result['need_aft_battery'])
                summary['saved_aft_battery'].append(result['saved_aft_battery'])
                
                for j in range(0, 26):
                    summary['n_'+str(j)].append(result['n_'+str(j)])   
                summary['npv'].append(result['npv'])
                df_result.to_csv(prefix + row['tableName'] + ".csv", index=False)
                if result['npv'] > 0:
                    c += 1
                    print(row['roof_area'], ', npv,', result['npv'], ',', c)
            else:
                summary['num_sc'].append(0)
                summary['total_installed_area'].append(0)
                summary['total_capacity'].append(0)
                summary['capex'].append(0)
                summary['opex'].append(0)
                summary['need_aft_battery'].append(0)
                summary['saved_aft_battery'].append(0)
                
                for j in range(0, 26):
                    summary['n_'+str(j)].append(0)   
                summary['npv'].append(0)
            
        print(i)  
        
    
    battery_df = pd.DataFrame.from_dict(summary)    
    battery_df.to_csv(outp, index=False)


In [ ]:

if __name__ == '__main__':
    parser = argparse.ArgumentParser()

    # add arguments to the parser
    parser.add_argument("inp") # time series csv file name
    parser.add_argument("outp") # npv result csv file name

    # parse the arguments
    args = parser.parse_args()
    main(args.inp, args.outp)